In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_finalproject")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlspchasiquiza01")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/products_v2.csv"

In [0]:
df_products = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
schema_products = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("availability", StringType(), True)
])


In [0]:
df_products_final = spark.read\
.option('header', True)\
.schema(schema_products)\
.csv(ruta)

In [0]:
products_final_df = df_products_final.withColumn("ingestion_date", current_timestamp())

In [0]:
products_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.products_bz")